<a href="https://colab.research.google.com/github/MayerT1/LiDAR_Dev/blob/main/Covatiate_tester.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
!pip install rasterio geopandas pandas torch torchvision scikit-learn einops

In [8]:
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [9]:
import os
import rasterio
import numpy as np
import pandas as pd

RASTER_DIR = '/content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/data'
eo_list = ['s1', 's2', 'landsat', 'landsat_idx', 'hls', 'tcap', 'dem', 's2_idx']

summary_data = []

def get_eo_type(filename):
    """Identify EO type based on filename prefix."""
    fname = filename.lower()
    for eo in eo_list:
        if fname.startswith(eo):
            return eo
    return 'unknown'

def scan_raster_files(directory):
    for fname in os.listdir(directory):
        if not fname.endswith('.tif'):
            continue

        path = os.path.join(directory, fname)
        try:
            with rasterio.open(path) as src:
                data = src.read(1)
                nodata_val = src.nodata

                nan_mask = np.isnan(data)
                nodata_mask = (data == nodata_val) if nodata_val is not None else np.zeros_like(data, dtype=bool)
                minus9999_mask = (data == -9999)

                total = data.size
                count_nan = np.sum(nan_mask)
                count_nodata = np.sum(nodata_mask)
                count_minus9999 = np.sum(minus9999_mask)

                eo_type = get_eo_type(fname)

                summary_data.append({
                    'filename': fname,
                    'eo_type': eo_type,
                    'nodata_metadata': nodata_val,
                    'total_pixels': total,
                    'NaN_count': count_nan,
                    'NaN_%': count_nan / total * 100,
                    '-9999_count': count_minus9999,
                    '-9999_%': count_minus9999 / total * 100,
                    'nodata_val_count': count_nodata,
                    'nodata_val_%': count_nodata / total * 100 if nodata_val is not None else 0
                })
        except Exception as e:
            print(f"[ERROR] Could not read {fname}: {e}")

scan_raster_files(RASTER_DIR)

# Convert to DataFrame
df = pd.DataFrame(summary_data)

# Save detailed file-level table
df.to_csv("raster_file_nan_summary.csv", index=False)

# === Filter problem files ===
df_problem = df[(df['NaN_count'] > 0) | (df['-9999_count'] > 0)]
df_problem.to_csv("raster_problem_files.csv", index=False)

# === Summary Stats Per EO Type ===
grouped = df.groupby('eo_type').agg({
    'filename': 'count',
    'total_pixels': 'sum',
    'NaN_count': 'sum',
    '-9999_count': 'sum',
    'nodata_val_count': 'sum'
}).reset_index()

grouped['NaN_%'] = grouped['NaN_count'] / grouped['total_pixels'] * 100
grouped['-9999_%'] = grouped['-9999_count'] / grouped['total_pixels'] * 100
grouped['nodata_val_%'] = grouped['nodata_val_count'] / grouped['total_pixels'] * 100

# Sort for clarity
grouped = grouped.sort_values(by='NaN_%', ascending=False)

# === Show Results ===
print("\n=== Per-EO Type Summary ===")
print(grouped.to_string(index=False))

print("\n=== Problem Files (NaN or -9999 present) ===")
print(df_problem[['filename', 'eo_type', 'NaN_%', '-9999_%']].to_string(index=False))

# Optional save
grouped.to_csv("eo_type_nan_summary.csv", index=False)



=== Per-EO Type Summary ===
eo_type  filename  total_pixels  NaN_count  -9999_count  nodata_val_count     NaN_%  -9999_%  nodata_val_%
    dem       167         23016      13779            0                 0 59.867049      0.0           0.0
    hls       168         23160        277            0                 0  1.196028      0.0           0.0
landsat       335         46176        534            0                 0  1.156445      0.0           0.0
   tcap       168         23160        267            0                 0  1.152850      0.0           0.0
     s1       168        185370       1710            0                 0  0.922479      0.0           0.0
     s2       336         98368        848            0                 0  0.862069      0.0           0.0

=== Problem Files (NaN or -9999 present) ===
                        filename eo_type      NaN_%  -9999_%
        hls_train_-8586_3519.tif     hls  12.121212      0.0
        hls_train_-8586_3523.tif     hls   9.090909   

xx

In [ ]:
# import os
# import rasterio
# import numpy as np

# RASTER_DIR = '/content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/data'

# def scan_raster_files_for_nodata(directory):
#     for fname in os.listdir(directory):
#         if fname.endswith('.tif'):
#             path = os.path.join(directory, fname)
#             with rasterio.open(path) as src:
#                 print(f"📂 Checking {fname}")

#                 data = src.read(1)  # read first band
#                 nodata_val = src.nodata
#                 nan_mask = np.isnan(data)
#                 nodata_mask = (data == nodata_val) if nodata_val is not None else np.zeros_like(data, dtype=bool)
#                 minus9999_mask = data == -9999

#                 total = data.size
#                 n_nan = np.sum(nan_mask)
#                 n_nodata = np.sum(nodata_mask)
#                 n_minus9999 = np.sum(minus9999_mask)

#                 print(f"   • Nodata value in metadata: {nodata_val}")
#                 print(f"   • NaNs found: {n_nan:,} ({n_nan / total:.2%})")
#                 print(f"   • '-9999' values found: {n_minus9999:,} ({n_minus9999 / total:.2%})")
#                 if nodata_val is not None:
#                     print(f"   • Pixels matching nodata value: {n_nodata:,} ({n_nodata / total:.2%})")
#                 print()

# scan_raster_files_for_nodata(RASTER_DIR)


Streaming output truncated to the last 5000 lines.
📂 Checking landsat_idx_train_-8587_3516.tif
   • Nodata value in metadata: None
   • NaNs found: 0 (0.00%)
   • '-9999' values found: 0 (0.00%)

📂 Checking landsat_idx_train_-8587_3518.tif
   • Nodata value in metadata: None
   • NaNs found: 0 (0.00%)
   • '-9999' values found: 0 (0.00%)

📂 Checking landsat_idx_train_-8587_3521.tif
   • Nodata value in metadata: None
   • NaNs found: 0 (0.00%)
   • '-9999' values found: 0 (0.00%)

📂 Checking landsat_idx_train_-8587_3522.tif
   • Nodata value in metadata: None
   • NaNs found: 0 (0.00%)
   • '-9999' values found: 0 (0.00%)

📂 Checking landsat_idx_train_-8588_3512.tif
   • Nodata value in metadata: None
   • NaNs found: 0 (0.00%)
   • '-9999' values found: 0 (0.00%)

📂 Checking landsat_idx_train_-8587_3523.tif
   • Nodata value in metadata: None
   • NaNs found: 0 (0.00%)
   • '-9999' values found: 0 (0.00%)

📂 Checking landsat_idx_train_-8588_3513.tif
   • Nodata value in metadata: None

In [ ]:
# import os
# import rasterio
# import numpy as np
# import pandas as pd

# RASTER_DIR = '/content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/data'

# summary_data = []

# def scan_raster_files(directory):
#     for fname in os.listdir(directory):
#         if not fname.endswith('.tif'):
#             continue

#         path = os.path.join(directory, fname)
#         with rasterio.open(path) as src:
#             data = src.read(1)
#             nodata_val = src.nodata

#             nan_mask = np.isnan(data)
#             nodata_mask = (data == nodata_val) if nodata_val is not None else np.zeros_like(data, dtype=bool)
#             minus9999_mask = (data == -9999)

#             total = data.size
#             count_nan = np.sum(nan_mask)
#             count_nodata = np.sum(nodata_mask)
#             count_minus9999 = np.sum(minus9999_mask)

#             summary_data.append({
#                 'filename': fname,
#                 'nodata_metadata': nodata_val,
#                 'total_pixels': total,
#                 'NaN_count': count_nan,
#                 'NaN_%': count_nan / total * 100,
#                 '-9999_count': count_minus9999,
#                 '-9999_%': count_minus9999 / total * 100,
#                 'nodata_val_count': count_nodata,
#                 'nodata_val_%': count_nodata / total * 100 if nodata_val is not None else 0
#             })

# scan_raster_files(RASTER_DIR)

# # Convert to DataFrame
# df = pd.DataFrame(summary_data)

# # Sort by percentage of NaNs or -9999s (if any)
# df_sorted = df.sort_values(by='NaN_%', ascending=False)

# # Show summary
# print("\n=== Per-File Summary (Top 10 Files with Highest NaN or -9999%) ===")
# print(df_sorted.head(100).to_string(index=False))

# # Global stats
# print("\n=== Overall Statistics ===")
# total_pixels = df['total_pixels'].sum()
# total_nans = df['NaN_count'].sum()
# total_9999s = df['-9999_count'].sum()
# total_nodata = df['nodata_val_count'].sum()

# print(f"• Total files checked: {len(df)}")
# print(f"• Total pixels analyzed: {total_pixels:,}")
# print(f"• Total NaNs: {total_nans:,} ({total_nans / total_pixels:.4%})")
# print(f"• Total -9999 values: {total_9999s:,} ({total_9999s / total_pixels:.4%})")
# print(f"• Total 'nodata' metadata values: {total_nodata:,} ({total_nodata / total_pixels:.4%})")

# # Optional: save as CSV
# # df_sorted.to_csv('dem_scan_summary.csv', index=False)



=== Per-File Summary (Top 10 Files with Highest NaN or -9999%) ===
                filename nodata_metadata  total_pixels  NaN_count      NaN_%  -9999_count  -9999_%  nodata_val_count  nodata_val_%
 dem_test_-8598_3517.tif            None           144        144 100.000000            0      0.0                 0             0
 dem_test_-8586_3516.tif            None           144        144 100.000000            0      0.0                 0             0
  dem_val_-8599_3514.tif            None           132        132 100.000000            0      0.0                 0             0
 dem_test_-8587_3519.tif            None           144        144 100.000000            0      0.0                 0             0
 dem_test_-8588_3515.tif            None           132        132 100.000000            0      0.0                 0             0
 dem_test_-8589_3520.tif            None           132        132 100.000000            0      0.0                 0             0
 dem_test_-8589